# Week 12: Recommendation Systems

**Goal:** Recommend a movie to a user.

Three methods: **Popularity**, **Content-based**, and **Collaborative filtering**.


In [1]:
# A small example: each row is one user–movie rating.
import pandas as pd

ratings = pd.DataFrame([
    ['Amy', 'Space Movie', 5],
    ['Amy', 'Robot Movie', 4],
    ['Amy', 'Love Movie', 1],
    ['Ben', 'Space Movie', 4],
    ['Ben', 'Robot Movie', 5],
    ['Ben', 'Comedy Movie', 4],
    ['Cara', 'Space Movie', 1],
    ['Cara', 'Love Movie', 5],
    ['Cara', 'Comedy Movie', 4]
], columns=['user', 'movie', 'rating'])

ratings

,user,movie,rating
0,Amy,Space Movie,5
1,Amy,Robot Movie,4
2,Amy,Love Movie,1
3,Ben,Space Movie,4
4,Ben,Robot Movie,5
5,Ben,Comedy Movie,4
6,Cara,Space Movie,1
7,Cara,Love Movie,5
8,Cara,Comedy Movie,4


### Our data

- Each row has a **user**, a **movie**, and a **rating**.
- Ratings go from **1 to 5**. Five means the user likes it a lot.


## 1. Popularity-based

Recommend movies with the **highest average rating**.


In [2]:
# Average rating for each movie, sorted from highest to lowest.
popular = (ratings.groupby('movie')['rating']
           .mean()
           .sort_values(ascending=False))
popular

movie
Robot Movie     4.500000
Comedy Movie    4.000000
Space Movie     3.333333
Love Movie      3.000000
Name: rating, dtype: float64

**Result:** Robot Movie is first, with an average rating of **4.5**.

**Problem:** Everyone gets the same list.


## 2. Content-based

Recommend movies that match a user’s **favorite genres**.


In [3]:
# Binary genre features: 1 = yes, 0 = no.
movies = pd.DataFrame({
    'movie': ['Space Movie', 'Robot Movie', 'Love Movie', 'Comedy Movie'],
    'sci_fi':  [1, 1, 0, 0],
    'romance': [0, 0, 1, 0],
    'comedy':  [0, 1, 0, 1]
})
movies

,movie,sci_fi,romance,comedy
0,Space Movie,1,0,0
1,Robot Movie,1,0,1
2,Love Movie,0,1,0
3,Comedy Movie,0,0,1


Here, **1 = yes** and **0 = no**. Robot Movie is both sci-fi and comedy.


In [4]:
# Example preference weights (not learned from the data).
# Amy likes science fiction and comedy, but not romance.
movies['score_for_amy'] = (
    movies['sci_fi'] * 2 +
    movies['romance'] * -2 +
    movies['comedy'] * 1
)
movies.sort_values('score_for_amy', ascending=False)

,movie,sci_fi,romance,comedy,score_for_amy
1,Robot Movie,1,0,1,3
0,Space Movie,1,0,0,2
3,Comedy Movie,0,0,1,1
2,Love Movie,0,1,0,-2


**Amy’s preferences:** sci-fi **+2**, romance **−2**, comedy **+1**. These are example scores.

**Result:** Robot Movie gets **3**, the highest score. Amy has already rated it, so a real system would usually remove it from new recommendations.


## 3. Collaborative filtering

Recommend movies liked by **people with similar tastes**.


In [5]:
# Turn long-form ratings into a user–item matrix.
rating_table = ratings.pivot(
    index='user', columns='movie', values='rating'
)
rating_table

movie,Comedy Movie,Love Movie,Robot Movie,Space Movie
user,,,,
Amy,NaN,1.0,4.0,5.0
Ben,4.0,NaN,5.0,4.0
Cara,4.0,5.0,NaN,1.0


This table shows **users as rows** and **movies as columns**. `NaN` means there is **no rating**.

Amy and Ben both like Space Movie and Robot Movie. We choose Ben as a similar user for this example.


In [6]:
# Simplified collaborative filtering: Ben is selected as similar by inspection.
# An unrated movie is not necessarily an unseen movie.
amy_movies = set(ratings.loc[ratings['user'] == 'Amy', 'movie'])

recommendations = ratings[
    (ratings['user'] == 'Ben') &
    (~ratings['movie'].isin(amy_movies)) &
    (ratings['rating'] >= 4)
]
recommendations

,user,movie,rating
5,Ben,Comedy Movie,4


**Result:** Ben rated **Comedy Movie = 4**, and Amy has **not rated** it.

**Recommend:** Comedy Movie to Amy.


## Quick review

| Method | Main idea |
|---|---|
| Popularity-based | Highest average ratings |
| Content-based | Match favorite genres |
| Collaborative filtering | Find similar users |
